# Preprocessing chung — Bộ dataset Train / Val / Test cho benchmark nhiều model

Notebook này **chỉ làm 1 việc**: chuẩn bị một bộ dữ liệu đã xử lý sẵn (resize + crop + chia
train/val/test) để **tất cả các notebook finetune model khác nhau dùng lại y nguyên**,
đảm bảo:

- Cùng tập ảnh train / validation / test (cùng seed, cùng tỉ lệ chia).
- Cùng cách resize/crop về kích thước vuông `RESOLUTION x RESOLUTION` (loại bỏ khác biệt
  do mỗi notebook tự xử lý ảnh khác nhau).
- Cùng caption/prompt cho mỗi ảnh.
- Test set là **held-out**, không được dùng để train hay chọn checkpoint ở bất kỳ model nào.

**Output** là một thư mục (và file `.zip`) chứa:

```
common_dataset_<style>/
├── train/*.png
├── val/*.png
├── test/*.png
├── manifest.csv          # split, image_path, original_path, prompt, ...
└── dataset_info.json      # cấu hình + thống kê để ghi log/báo cáo
```

Các notebook finetune chỉ cần đọc `manifest.csv` (xem class `CommonStyleDataset` ở cuối
notebook) — **không tự resize/crop/split lại nữa**.

> Nếu các model trong benchmark cần resolution khác nhau (ví dụ SD v1.5 = 512, SDXL = 1024),
> đặt `RESOLUTION` theo resolution lớn nhất cần dùng, hoặc chạy lại notebook này với
> `RESOLUTION` khác — vì `original_path` được giữ trong manifest, cột `split` sẽ **không đổi**
> giữa các lần chạy (cùng `SEED`), nên việc so sánh vẫn công bằng.


In [ ]:
# =========================
# IMPORT
# =========================

import os, json, random, shutil, hashlib
from pathlib import Path

import pandas as pd
from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm
from IPython.display import display

print("OK")


In [ ]:
# =========================
# CẤU HÌNH CHUNG
# =========================

# ---- Dữ liệu nguồn ----
STYLE_NAME = "Impressionism"
DATA_ROOT  = "/kaggle/input/datasets/steubk/wikiart/Impressionism"

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# ---- Resize/crop CHUNG cho mọi model ----
# Mọi ảnh sẽ được resize cạnh ngắn về RESOLUTION rồi center-crop thành RESOLUTION x RESOLUTION.
RESOLUTION = 512   # đổi nếu các model trong benchmark cần resolution khác (ví dụ SDXL = 1024)

# ---- Chia train / val / test ----
VAL_RATIO  = 0.10
TEST_RATIO = 0.10   # held-out, KHÔNG dùng để train hoặc chọn checkpoint ở bất kỳ model nào

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)

# ---- Caption / prompt chung cho toàn bộ ảnh ----
TRIGGER_TOKEN = "skstyle"
STYLE_PROMPT  = f"{TRIGGER_TOKEN} impressionism style painting"

# ---- Giới hạn dataset (debug nhanh) ----
MAX_DATASET_IMAGES = None   # None = toàn bộ; đặt số nhỏ (vd 300) để test nhanh pipeline

# ---- Output ----
OUTPUT_DIR = f"/kaggle/working/common_dataset_{STYLE_NAME.lower()}"

print("STYLE_NAME:", STYLE_NAME)
print("RESOLUTION:", RESOLUTION)
print("VAL_RATIO / TEST_RATIO:", VAL_RATIO, "/", TEST_RATIO)
print("STYLE_PROMPT:", STYLE_PROMPT)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# =========================
# TÌM VÀ LIỆT KÊ ẢNH NGUỒN
# =========================

def count_images(folder):
    p = Path(folder)
    return 0 if not p.exists() else sum(
        1 for f in p.rglob("*") if f.is_file() and f.suffix.lower() in IMG_EXTS
    )

def find_style_folders(style_name):
    base = Path("/kaggle/input")
    if not base.exists():
        return []
    matches = []
    for folder in base.rglob("*"):
        if folder.is_dir() and folder.name.lower() == style_name.lower():
            n = count_images(str(folder))
            if n > 0:
                matches.append((folder, n))
    return matches

if count_images(DATA_ROOT) == 0:
    print("DATA_ROOT không hợp lệ, tự động tìm...")
    matches = find_style_folders(STYLE_NAME)
    assert len(matches) > 0, f"Không tìm thấy '{STYLE_NAME}' trong /kaggle/input"
    DATA_ROOT = str(max(matches, key=lambda x: x[1])[0])

assert Path(DATA_ROOT).name.lower() == STYLE_NAME.lower(), (
    f"DATA_ROOT='{Path(DATA_ROOT).name}' không khớp STYLE_NAME='{STYLE_NAME}'"
)

all_image_files = sorted([
    p for p in Path(DATA_ROOT).rglob("*")
    if p.is_file() and p.suffix.lower() in IMG_EXTS
])
assert len(all_image_files) > 0, f"Không tìm thấy ảnh nào trong {DATA_ROOT}"

print(f"Tìm thấy {len(all_image_files)} ảnh trong: {DATA_ROOT}")


In [ ]:
# =========================
# KIỂM TRA ẢNH HỎNG / QUÁ NHỎ
# =========================

rng = random.Random(SEED)
rng.shuffle(all_image_files)

if MAX_DATASET_IMAGES is not None:
    all_image_files = all_image_files[:MAX_DATASET_IMAGES]
    print(f"[DEBUG] Giới hạn còn {len(all_image_files)} ảnh")

MIN_SIDE = 64  # bỏ ảnh quá nhỏ so với RESOLUTION sẽ làm upsample mạnh, gây mờ

valid_image_files, bad_image_files = [], []
for p in tqdm(all_image_files, desc="Kiểm tra ảnh"):
    try:
        with Image.open(p) as img:
            img.convert("RGB")
            w, h = img.size
        if min(w, h) < MIN_SIDE:
            bad_image_files.append((p, f"too small ({w}x{h})"))
            continue
        valid_image_files.append(p)
    except Exception as e:
        bad_image_files.append((p, str(e)))

print(f"Hợp lệ: {len(valid_image_files)} | Lỗi/bỏ qua: {len(bad_image_files)}")
assert len(valid_image_files) >= 10, "Quá ít ảnh hợp lệ để chia train/val/test."

if bad_image_files:
    bad_df = pd.DataFrame([{"path": str(p), "reason": r} for p, r in bad_image_files])
    print(bad_df.head(10))


In [ ]:
# =========================
# CHIA TRAIN / VAL / TEST (cố định theo SEED)
# =========================

usable = list(valid_image_files)
random.Random(SEED).shuffle(usable)

n_total = len(usable)
n_test  = max(1, int(n_total * TEST_RATIO))
n_val   = max(1, int(n_total * VAL_RATIO))
n_train = n_total - n_val - n_test
assert n_train > 0, "Không đủ ảnh để chia train/val/test với tỉ lệ hiện tại."

test_files  = usable[:n_test]
val_files   = usable[n_test : n_test + n_val]
train_files = usable[n_test + n_val :]

assert not (set(train_files) & set(val_files))
assert not (set(train_files) & set(test_files))
assert not (set(val_files)   & set(test_files))

print(f"Tổng số ảnh hợp lệ: {n_total}")
print(f"Train: {len(train_files)} | Val: {len(val_files)} | Test (held-out): {len(test_files)}")

if len(test_files) < 100:
    print("[CẢNH BÁO] Test set < 100 ảnh — FID/KID (nếu dùng) chỉ mang tính tham khảo, "
          "chủ yếu để kiểm tra pipeline.")


In [ ]:
# =========================
# HÀM TIỀN XỬ LÝ ẢNH (resize + center-crop)
# =========================

# Đây là bước tiền xử lý DUY NHẤT và CHUNG cho mọi model -> đảm bảo công bằng.
# KHÔNG áp dụng augmentation ngẫu nhiên (random flip, color jitter, ...) ở đây.
# Nếu model nào cần augmentation, hãy thêm ở notebook finetune riêng, áp dụng
# TRÊN ẢNH ĐÃ XỬ LÝ NÀY (xem class CommonStyleDataset ở cuối notebook).

preprocess = transforms.Compose([
    transforms.Resize(RESOLUTION, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(RESOLUTION),
])

def process_and_save(src_path: Path, dst_path: Path):
    with Image.open(src_path) as img:
        img = img.convert("RGB")
        orig_w, orig_h = img.size
        out = preprocess(img)
        out.save(dst_path, format="PNG")
    return orig_w, orig_h


In [ ]:
# =========================
# XỬ LÝ TOÀN BỘ ẢNH CHO 3 SPLIT + TẠO MANIFEST
# =========================

splits = {"train": train_files, "val": val_files, "test": test_files}

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
    print("Đã xóa output cũ:", OUTPUT_DIR)

manifest_rows = []

for split_name, files in splits.items():
    split_dir = Path(OUTPUT_DIR) / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    for idx, src in enumerate(tqdm(files, desc=f"Xử lý {split_name}")):
        # tên file mới ổn định, tránh ký tự đặc biệt / unicode trong path gốc
        stem = hashlib.md5(str(src).encode("utf-8")).hexdigest()[:16]
        dst = split_dir / f"{split_name}_{idx:05d}_{stem}.png"

        orig_w, orig_h = process_and_save(src, dst)

        manifest_rows.append({
            "split": split_name,
            "image_path": str(dst.relative_to(OUTPUT_DIR)),
            "original_path": str(src),
            "original_width": orig_w,
            "original_height": orig_h,
            "prompt": STYLE_PROMPT,
        })

manifest_df = pd.DataFrame(manifest_rows)
print(manifest_df["split"].value_counts())


In [ ]:
# =========================
# LƯU manifest.csv + dataset_info.json
# =========================

manifest_path = os.path.join(OUTPUT_DIR, "manifest.csv")
manifest_df.to_csv(manifest_path, index=False)

dataset_info = {
    "style_name": STYLE_NAME,
    "data_root": DATA_ROOT,
    "resolution": RESOLUTION,
    "seed": SEED,
    "val_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "trigger_token": TRIGGER_TOKEN,
    "style_prompt": STYLE_PROMPT,
    "num_total_valid": len(usable),
    "num_train": len(train_files),
    "num_val": len(val_files),
    "num_test": len(test_files),
    "num_bad_skipped": len(bad_image_files),
    "note": (
        "Bo dataset preprocessing CHUNG dung cho tat ca model trong benchmark. "
        "Moi notebook finetune phai doc dung manifest.csv nay (cung split, cung anh "
        "da resize/crop, cung prompt) de dam bao so sanh cong bang. "
        "Split 'test' la held-out, KHONG dung de train hoac chon checkpoint."
    ),
}

with open(os.path.join(OUTPUT_DIR, "dataset_info.json"), "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(json.dumps(dataset_info, ensure_ascii=False, indent=2))
print("\nManifest:", manifest_path)


In [ ]:
# =========================
# KIỂM TRA NHANH (sanity check)
# =========================

# 1) Xem 1 ảnh mẫu mỗi split
for split_name in ["train", "val", "test"]:
    sample = manifest_df[manifest_df["split"] == split_name].iloc[0]
    img = Image.open(os.path.join(OUTPUT_DIR, sample["image_path"]))
    print(f"{split_name}: size={img.size}, prompt='{sample['prompt']}'")
    display(img)

# 2) Đảm bảo không có ảnh gốc nào bị trùng giữa các split
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    sa = set(manifest_df[manifest_df["split"] == a]["original_path"])
    sb = set(manifest_df[manifest_df["split"] == b]["original_path"])
    overlap = sa & sb
    assert not overlap, f"Trùng ảnh giữa {a} và {b}: {len(overlap)} ảnh"

print("\nKhông có ảnh trùng giữa train/val/test. OK.")


In [ ]:
# =========================
# ĐÓNG GÓI OUTPUT ĐỂ DÙNG CHO CÁC NOTEBOOK FINETUNE KHÁC
# =========================

zip_path = shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
print("Đã đóng gói:", zip_path)
print(f"Kích thước: {os.path.getsize(zip_path) / 1e6:.1f} MB")

print("\nNội dung OUTPUT_DIR:")
for p in sorted(Path(OUTPUT_DIR).iterdir()):
    print(" -", p.name)


## Cách dùng trong các notebook finetune model

1. Copy/upload thư mục `common_dataset_<style>/` (hoặc giải nén file `.zip`) vào notebook
   finetune của từng model.
2. Đọc `manifest.csv` và dùng class `CommonStyleDataset` dưới đây (copy nguyên cell này
   sang notebook finetune) để tạo `train_dataset`, `val_dataset`, `test_dataset`.
3. Mọi model dùng **đúng `STYLE_PROMPT`** trong `dataset_info.json` làm caption/prompt
   huấn luyện (hoặc trigger token tương ứng), trừ khi việc so sánh caption khác nhau là
   một phần chủ đích của benchmark.
4. **Không** dùng `test` split để train hay để chọn checkpoint — chỉ dùng cho đánh giá
   cuối cùng (ví dụ FID/KID giữa base model và model đã finetune).


In [ ]:
# =========================
# CLASS DATASET DÙNG CHUNG CHO CÁC NOTEBOOK FINETUNE
# (copy cell này sang notebook finetune của từng model)
# =========================

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from pathlib import Path
from PIL import Image


class CommonStyleDataset(Dataset):
    """
    Đọc dữ liệu từ bộ dataset chung (manifest.csv) đã được tiền xử lý sẵn:
    resize + center-crop về RESOLUTION x RESOLUTION, chia sẵn train/val/test.

    Ở đây chỉ cần ToTensor + Normalize (+ augmentation tuỳ chọn cho train),
    KHÔNG resize/crop lại để giữ ảnh giống nhau giữa các model.
    """

    def __init__(self, manifest_csv, dataset_root, split, is_train=True, use_flip=True):
        df = pd.read_csv(manifest_csv)
        self.df = df[df["split"] == split].reset_index(drop=True)
        assert len(self.df) > 0, f"Split '{split}' trống."
        self.dataset_root = Path(dataset_root)

        t = []
        if is_train and use_flip:
            t.append(transforms.RandomHorizontalFlip(p=0.5))
        t += [
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ]
        self.transform = transforms.Compose(t)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.dataset_root / row["image_path"]
        with Image.open(img_path) as img:
            pv = self.transform(img.convert("RGB"))
        return {"pixel_values": pv, "text": row["prompt"], "path": str(img_path)}


# ---- Ví dụ sử dụng trong notebook finetune ----
# MANIFEST_PATH = os.path.join(OUTPUT_DIR, "manifest.csv")
#
# train_dataset = CommonStyleDataset(MANIFEST_PATH, OUTPUT_DIR, split="train", is_train=True)
# val_dataset   = CommonStyleDataset(MANIFEST_PATH, OUTPUT_DIR, split="val",   is_train=False)
# test_dataset  = CommonStyleDataset(MANIFEST_PATH, OUTPUT_DIR, split="test",  is_train=False)
#
# train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)
# val_dataloader   = DataLoader(val_dataset,   batch_size=1, shuffle=False)
# test_dataloader  = DataLoader(test_dataset,  batch_size=1, shuffle=False)
#
# print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
